# NEXUS Vision on a free GPU

This trains your own NEXUS Vision, the model NEXUS Live uses to see your screen and choose the next click or key,
and puts it in a **private** repo on your Hugging Face account. Then your laptop downloads it once and runs it offline.

1. **Runtime > Change runtime type > T4 GPU** (on Kaggle: Settings > Accelerator > GPU T4, and turn Internet on).
2. Run the cells in order. With the settings below it takes about two hours on a free T4.

It fine-tunes Qwen3-VL-2B-Instruct (Apache-2.0) on Windows screenshots from OS-Atlas (Apache-2.0) and, if you add them,
your own teach recordings, then tests the result against the base model on examples it never saw.

In [ ]:
!pip -q install "nexus-ai[vision] @ git+https://github.com/nagu-io/nexus@main"

## Log in to Hugging Face

The model is uploaded with a token that can write: make one at https://huggingface.co/settings/tokens.
Store it as a secret named `HF_TOKEN` (the key icon on the left in Colab; Add-ons > Secrets on Kaggle),
or paste it into the box this cell shows. Never paste it into a code cell.

In [ ]:
import os

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = _secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
else:
    from huggingface_hub import notebook_login
    notebook_login()

## Settings

`REPO` is where the model goes on your account. `TEACH` is optional: your teach recordings, uploaded as a private
dataset repo (see docs/NEXUS_VISION.md). `LIMIT` is how many OS-Atlas elements to learn from; 2000 fits a free session.

In [ ]:
REPO = "YOUR_NAME/nexus-vision"
TEACH = ""
LIMIT = 2000

In [ ]:
from nexus.live.build import main

args = ["--systems", "windows", "--limit", str(LIMIT), "--push", REPO,
        "--batch-size", "1", "--grad-accum", "16", "--eval-limit", "100"]
if TEACH:
    args += ["--teach", TEACH]
main(args)

## On your laptop

```
pip install -e .[live,vision]
nexus vision pull YOUR_NAME/nexus-vision
nexus live --voice
```